# EWS Fraud Detection - Tahap 2A: PDF Inventory

Jupyter Notebook ini digunakan untuk mendata seluruh berkas PDF hasil unduhan dari direktori Google Drive (`G:\My Drive\Dataset PUI-PT\IDX_Downloads`) dan memetakan lokasinya. Inventaris ini akan digunakan sebagai basis data pada tahap selanjutnya dalam mendeteksi fraud (Fraud Detection).

In [1]:
import os
import pandas as pd
from tqdm.notebook import tqdm

# Jalur utama direktori unduhan dan file rekap
ROOT_DIR = r"G:\My Drive\Dataset PUI-PT\IDX_Downloads"
excel_path = r"../Data_Crawling/Rekap_Perusahaan_2021_2024.xlsx"

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"Excel Rekap: {excel_path}")

ROOT_DIR: G:\My Drive\Dataset PUI-PT\IDX_Downloads
Excel Rekap: ../Data_Crawling/Rekap_Perusahaan_2021_2024.xlsx


In [2]:
# 1. Muat perusahaan yang Valid dari Excel Rekap
try:
    df_companies = pd.read_excel(excel_path)
    df_companies.columns = [c.strip() for c in df_companies.columns]
    df_valid = df_companies[df_companies['Status'].astype(str).str.strip().str.lower() == 'valid'].copy()
    df_valid['Kode'] = df_valid['Kode'].astype(str).str.strip()
    valid_tickers = set(df_valid['Kode'])
    print(f"Berhasil memuat daftar perusahaan. Total emiten Valid: {len(valid_tickers)}")
except Exception as e:
    print(f"Peringatan: Gagal memuat Excel Rekap ({e}). Seluruh emiten di folder unduhan akan dipindai.")
    valid_tickers = None

Peringatan: Gagal memuat Excel Rekap ([Errno 2] No such file or directory: '../Data_Crawling/Rekap_Perusahaan_2021_2024.xlsx'). Seluruh emiten di folder unduhan akan dipindai.


In [3]:
# 2. Proses Pembuatan PDF Inventory (Pemindaian Direktori)
records = []

if not os.path.exists(ROOT_DIR):
    print(f"ERROR: Folder {ROOT_DIR} tidak ditemukan!")
else:
    # Dapatkan semua folder emiten di ROOT_DIR
    all_folders = [f for f in os.listdir(ROOT_DIR) if os.path.isdir(os.path.join(ROOT_DIR, f))]
    
    # Filter emiten yang Valid (jika daftar valid_tickers tersedia)
    if valid_tickers is not None:
        emiten_to_scan = [f for f in all_folders if f in valid_tickers]
    else:
        emiten_to_scan = all_folders
        
    emiten_to_scan = sorted(emiten_to_scan)
    print(f"Memulai pemindaian untuk {len(emiten_to_scan)} folder emiten...")
    
    # Loop dengan progress bar tqdm
    for kode in tqdm(emiten_to_scan, desc="Scanning Emiten"):
        kode_path = os.path.join(ROOT_DIR, kode)
        
        for tahun in os.listdir(kode_path):
            tahun_path = os.path.join(kode_path, tahun)
            if not os.path.isdir(tahun_path):
                continue
                
            for kategori in [
                "Laporan_Keuangan",
                "Laporan_Tahunan",
                "Dokumen_Lainnya"
            ]:
                kategori_path = os.path.join(tahun_path, kategori)
                if not os.path.exists(kategori_path):
                    continue
                    
                # Gunakan os.walk untuk mencari file PDF secara mendalam (deep scan)
                for root, dirs, files in os.walk(kategori_path):
                    for file in files:
                        if file.lower().endswith(".pdf"):
                            file_path = os.path.join(root, file)
                            try:
                                file_size = os.path.getsize(file_path)
                            except:
                                file_size = 0
                                
                            # Hanya masukkan file valid (>1 KB)
                            if file_size > 1000:
                                records.append({
                                    "kode": kode,
                                    "tahun": tahun,
                                    "kategori": kategori,
                                    "file": file,
                                    "path": file_path
                                })

df = pd.DataFrame(records)
print(f"Selesai memindai. Ditemukan {len(df)} file PDF.")

ERROR: Folder G:\My Drive\Dataset PUI-PT\IDX_Downloads tidak ditemukan!
Selesai memindai. Ditemukan 0 file PDF.


In [4]:
# 3. Ekspor Hasil Inventaris ke Excel
if not df.empty:
    df.to_excel("PDF_Inventory.xlsx", index=False)
    print(f"Inventaris PDF berhasil disimpan di 'PDF_Inventory.xlsx'")
    print(f"Bentuk data (shape): {df.shape}")
else:
    print("Peringatan: Tidak ada data PDF yang ditemukan atau diindeks.")

Peringatan: Tidak ada data PDF yang ditemukan atau diindeks.
